In [1]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dropout, Dense
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping


In [2]:
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [3]:
data = gutenberg.raw('austen-emma.txt')
with open('emma.txt','w') as f:
  f.write(data)

In [4]:
data = gutenberg.raw('austen-emma.txt')
with open('emma.txt', 'r') as f:
  text = f.read().lower()

In [5]:
##tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1
total_words

7233

In [6]:
#index
tokenizer.word_index

{'to': 1,
 'the': 2,
 'and': 3,
 'of': 4,
 'i': 5,
 'a': 6,
 'it': 7,
 'her': 8,
 'was': 9,
 'she': 10,
 'in': 11,
 'not': 12,
 'you': 13,
 'be': 14,
 'he': 15,
 'that': 16,
 'had': 17,
 'but': 18,
 'as': 19,
 'for': 20,
 'have': 21,
 'is': 22,
 'with': 23,
 'very': 24,
 'mr': 25,
 'his': 26,
 'at': 27,
 'so': 28,
 'all': 29,
 'could': 30,
 'would': 31,
 'emma': 32,
 'him': 33,
 'been': 34,
 'no': 35,
 'my': 36,
 'mrs': 37,
 'on': 38,
 'any': 39,
 'do': 40,
 'miss': 41,
 'were': 42,
 'me': 43,
 'will': 44,
 'by': 45,
 'must': 46,
 'which': 47,
 'there': 48,
 'from': 49,
 'they': 50,
 'what': 51,
 'this': 52,
 'or': 53,
 'such': 54,
 'much': 55,
 'if': 56,
 'said': 57,
 'more': 58,
 'an': 59,
 'are': 60,
 'one': 61,
 'them': 62,
 'every': 63,
 'than': 64,
 'harriet': 65,
 'am': 66,
 'well': 67,
 'thing': 68,
 'weston': 69,
 'think': 70,
 'how': 71,
 'should': 72,
 'your': 73,
 'when': 74,
 'little': 75,
 'being': 76,
 'never': 77,
 'good': 78,
 'we': 79,
 'knightley': 80,
 'did': 81,
 '

In [7]:
#input sequence
input_sequences=[]
for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1,len(token_list)):
      ngram_sequence = token_list[:i+1]
      input_sequences.append(ngram_sequence)



In [8]:
input_sequences

[[32, 45],
 [32, 45, 92],
 [32, 45, 92, 4410],
 [32, 45, 92, 4410, 4411],
 [2794, 5],
 [346, 5],
 [32, 96],
 [32, 96, 493],
 [32, 96, 493, 633],
 [32, 96, 493, 633, 3],
 [32, 96, 493, 633, 3, 1024],
 [32, 96, 493, 633, 3, 1024, 23],
 [32, 96, 493, 633, 3, 1024, 23, 6],
 [32, 96, 493, 633, 3, 1024, 23, 6, 532],
 [32, 96, 493, 633, 3, 1024, 23, 6, 532, 163],
 [3, 171],
 [3, 171, 697],
 [3, 171, 697, 156],
 [3, 171, 697, 156, 1],
 [3, 171, 697, 156, 1, 2795],
 [3, 171, 697, 156, 1, 2795, 97],
 [3, 171, 697, 156, 1, 2795, 97, 4],
 [3, 171, 697, 156, 1, 2795, 97, 4, 2],
 [3, 171, 697, 156, 1, 2795, 97, 4, 2, 238],
 [3, 171, 697, 156, 1, 2795, 97, 4, 2, 238, 1853],
 [4, 1551],
 [4, 1551, 3],
 [4, 1551, 3, 17],
 [4, 1551, 3, 17, 675],
 [4, 1551, 3, 17, 675, 1025],
 [4, 1551, 3, 17, 675, 1025, 588],
 [4, 1551, 3, 17, 675, 1025, 588, 61],
 [4, 1551, 3, 17, 675, 1025, 588, 61, 364],
 [4, 1551, 3, 17, 675, 1025, 588, 61, 364, 11],
 [4, 1551, 3, 17, 675, 1025, 588, 61, 364, 11, 2],
 [4, 1551, 3, 1

In [9]:
max_len = max([len(x) for x in input_sequences])
max_len

17

In [10]:
input_sequences = np.array(pad_sequences(input_sequences,maxlen = max_len,padding = 'pre'))
input_sequences

array([[   0,    0,    0, ...,    0,   32,   45],
       [   0,    0,    0, ...,   32,   45,   92],
       [   0,    0,    0, ...,   45,   92, 4410],
       ...,
       [   0,    0,    0, ...,  534,  260,    4],
       [   0,    0,    0, ...,  260,    4,    2],
       [   0,    0,    0, ...,    4,    2, 2784]], dtype=int32)

In [11]:
# predictors and label
X,y = input_sequences[:,: -1],input_sequences[:,-1]

In [12]:
X

array([[  0,   0,   0, ...,   0,   0,  32],
       [  0,   0,   0, ...,   0,  32,  45],
       [  0,   0,   0, ...,  32,  45,  92],
       ...,
       [  0,   0,   0, ...,   2, 534, 260],
       [  0,   0,   0, ..., 534, 260,   4],
       [  0,   0,   0, ..., 260,   4,   2]], dtype=int32)

In [13]:
y

array([  45,   92, 4410, ...,    4,    2, 2784], dtype=int32)

In [14]:
y=to_categorical(y,num_classes = total_words)
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [15]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [16]:
model = Sequential([
    Input(shape=(max_len - 1,)),
    Embedding(input_dim=total_words, output_dim=100),
    LSTM(150, return_sequences=True),
    Dropout(0.2),
    LSTM(100),
    Dense(total_words, activation="softmax")
])



In [17]:
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 16, 100)        │       723,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 16, 150)        │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 150)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 7233)           │       730,533 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,704,833 (6.50 MB)

 Trainable params: 1,704,833 (6.50 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
early_stop = EarlyStopping( monitor="val_loss",patience=5,restore_best_weights=True,
    verbose=1
)

In [ ]:
history = model.fit(
    x_train,
    y_train,
    epochs=50,
    batch_size=128,
    validation_data=(x_test, y_test),
    callbacks=[early_stop]
)